# **Permutation Test for Differential Overlap Between IFIT3 (or YTH) and ADAR-Ctl TRIBE Sites Versus other TRIBE data and published CLIP-seq data**

## Disclaimer: ChatGPT was used in part to generate this analyses scheme and code

### **Purpose**
This notebook tests whether overlap between IFIT3-ADAR (or ADAR-YTH) HyperTRIBE edits with other data sets is significantly different (enriched/depleted) from the corresponding Control data, using different windowing strategies and an empirical permutation test. This notebook describes a comparison using multiple window sizes to compare IFIT3-ADAR and ADAR-YTH HyperTRIBE edits (as in Fig 2D), but was also applied to published CLIP-seq data as in Fig 1E, 1F, and 2C.

---

## **Overview**

## **Pre-process input data**
1. **Generate whitelist bed**
2. **Parse and filter published data against whitelist**

## **HyperTRIBE edit processing and overlap analyses**
1. **Generate clusters based on increasing window sizes**
2. **Merge clusters using the bedtools groupby function**
3. **Add flanking intervals corresponding to the clustering distance in step 1**
4. **Compute observed overlaps between test and control data then difference**
5. **Run permutation test: shuffle peaks, recompute overlaps, build null distribution**
6. **Empirical p-values and multiple testing correction**
7. **Summary and visualizations**

## **Input data origins**

- GRCh38.refflat : Downloaded from UCSC table browser
    - Same that was used for HyperTRIBE-seq analyses
- postar3 bed: Downloaded directly from postar3 CLIPDB via email request (PMID: 25652745)
- IFIT2 data: downloaded from publications and filtered/sorted against whitelist
    - Tran et al. 2020 (PMID: 32839537): Data was formatted to bed from supplement tables
    - Luo et al. 2020 (PMID: 32807991): IFIT2 data was downloaded from publication and filtered for peaks common between reps with l10p>=2, l2fc>=2. Converted to hg38 with UCSC liftover tool
- meRIP data: Gokhale et al. 2020 (PMID: 31810760). Data downloaded from publication. Used all 31647 peaks filtered against whitelist.
- IFIT3 bed: From HyperTRIBE seq analyses described in this repo. Sites with ≥8% edit, ≥0.4 LFC, padj<0.05 (or NA)
- control bed: From matched IFIT3-ADAR HyperTRIBE seq analyses described in this repo. Sites with ≥8% edit, <0.4 LFC
- genome sizes: From STAR index of GRCh38 used for alignment of seq data with non-standard chromosomes removed
- TRIBE_Huh7_whitelist.txt: Derived from genecounts table generated during alignment

## **Preprocess inputs**

In [ ]:
import pandas as pd
import numpy as np
import subprocess
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
N_PERMUT = 1000  # Number of shuffles for the permutation test
N_CPUS = 4
GENE_BED = "Inputs/gene_bed.bed"
WHITELIST_TXT = "Inputs/TRIBE_Huh7_whitelist.txt"
REFFLAT = "Inputs/GRCh38.refflat"
GENE_COUNTS = "Inputs/salmon.merged.gene_counts.tsv"
IFIT3_BED = "Inputs/IFIT3_TRIBE_HITS.sorted.bed"
CONTROL_BED = "Inputs/IFIT3_TRIBE_NS.sorted.bed"

CLIPDB = "Inputs/POSTAR3.bed"
FILTERED_CLIPDB = "Inputs/POSTAR3.filter.bed"

RBP_DIR = Path("Inputs/rbp_beds")   # Input RBP peaks
FILTERED_RBP_DIR = Path("Inputs/filtered_RBPs")
TEST_RBP_DIR = Path("Inputs/test_RBPs")
GENOME_SIZES = "Inputs/chrNameLength.txt"
WHITELIST_BED = "Inputs/TRIBE_Huh7_whitelist.bed"

PERMUT_DIR = Path("shuffled")
PERMUT_DIR.mkdir(exist_ok=True)

os.makedirs(FILTERED_RBP_DIR, exist_ok=True)

# Base functions
def count_lines(filename):
    with open(filename, 'r') as file:
        return sum(1 for _ in file)

## 1. **Prepare whitelist gene BED**

In [ ]:
# convert the reflat into bed format

# Read the refFlat file (skip the comment/header line)
df = pd.read_csv(REFFLAT, sep='\s+', comment='#', header=None,
                 names=["geneName","name","chrom","strand","txStart","txEnd","cdsStart","cdsEnd",
                        "exonCount","exonStarts","exonEnds"])

# Make a BED DataFrame: chrom, txStart, txEnd, geneName (the basic gene interval)
bed_df = df[['chrom', 'txStart', 'txEnd', 'geneName']].drop_duplicates()

# Save as BED (no header, tab-separated, no index)
bed_df.to_csv('Inputs/gene_bed.bed', sep='\t', header=False, index=False)

# Also make one with intervals merged when gene name is the same
bed_collapsed = bed_df.groupby(['chrom', 'geneName']).agg({'txStart':'min', 'txEnd':'max'}).reset_index()
bed_collapsed = bed_collapsed[['chrom', 'txStart', 'txEnd', 'geneName']]
bed_collapsed.to_csv(GENE_BED, sep='\t', header=False, index=False)

In [ ]:
# filter gene counts table for genes that have ≥10 counts in all replicates

# Read the table
df = pd.read_csv(GENE_COUNTS, sep='\t')

# Select columns 2 to 10 (0-based: gene_name is col 1, ctl_1 is col 2, ... yth_3 is col 10)
cols_of_interest = df.columns[2:11]  # df.columns[2] up to df.columns[10] inclusive

# Check if **all** values in each row in these columns are >=10
mask = (df[cols_of_interest].astype(float) >= 10).all(axis=1)

# Output gene names (from gene_name column, which is col 1, index 1)
df.loc[mask, 'gene_name'].to_csv(WHITELIST_TXT, index=False, header=False)

In [ ]:
# Create Whitelist bed

with open(WHITELIST_TXT) as wfile:
    whitelist_genes = {line.strip() for line in wfile}
print(f"Whitelist contains {len(whitelist_genes)} genes.")

# Now filter gene_bed.bed by gene name (col 3)
genes_df = pd.read_csv(GENE_BED, sep="\t", header=None)
genes_df_filtered = genes_df[genes_df[3].isin(whitelist_genes)]
genes_df_filtered.to_csv(WHITELIST_BED, sep="\t", header=False, index=False)
print(f"Whitelist gene BED written to {WHITELIST_BED}, {genes_df_filtered.shape[0]} intervals.")

## 2. **Parse and filter published data against whitelist**

In [ ]:
# Use BEDTools intersect to keep genes that were expressed in HyperTRIBE-seq data
subprocess.run([
    "bedtools", "intersect", "-u",
    "-a", str(CLIPDB),
    "-b", WHITELIST_BED
    ], stdout=open(FILTERED_CLIPDB, "w"))
print(f"CLIPdb filtered from {count_lines(CLIPDB)} intervals to {count_lines(FILTERED_CLIPDB)}")

### Parse the large CLIPdb bed into individual beds for each RBP and sort them

In [ ]:
# Used chunked reading to avoid memory issues
reader = pd.read_csv(FILTERED_CLIPDB, sep='\t', header=None, names=[
    "chrom","start","end","rbp","score","strand","cell","mode","sample","GEO","other"], chunksize=1_000_000)

temp_dir = Path("Inputs/split_rbp_temps")
temp_dir.mkdir(exist_ok=True)

# Split into rbp-specific files (one per RBP)
rbp_files = dict()

for chunk in reader:
    for name, group in chunk.groupby('rbp'):
        fname = temp_dir / f"{name}.bed"
        # Append chunk to a file for each RBP
        with open(fname, 'a') as f:
            group.iloc[:, :7].to_csv(f, sep='\t', header=False, index=False)
        rbp_files[name] = fname

print(f"Split finished, found {len(rbp_files)} RBPs")

In [ ]:
FILTERED_RBP_DIR.mkdir(exist_ok=True)

for rbp, fname in rbp_files.items():
    filtered_bed = FILTERED_RBP_DIR / f"{rbp}.bed"
    # Filter with bedtools intersect and sort with Unix sort
    with open(filtered_bed, "w") as out:
        sort = subprocess.Popen(
            ["sort", "-k1,1", "-k2,2n", str(fname)],
            stdout=out
        )
        sort.communicate()
print("All individual RBP beds sorted.")

# Optional: Clean up temp files to save space
for f in temp_dir.glob("*.bed"):
    os.remove(f)
temp_dir.rmdir()

## **HyperTRIBE edit processing and overlap analyses**

In [35]:
import os
from pathlib import Path
import subprocess
import re
import pandas as pd
%load_ext autoreload
%autoreload 2
import overlap_scripts

wd = "/cwork/mt354/MGT494_overlaps_v2/"

# Configuration
N_PERMUT = 1000  # Number of shuffles for the permutation test
N_CPUS = 4
GENE_BED = "Inputs/gene_bed.bed"
REFFLAT = "Inputs/GRCh38.refflat"
GENE_COUNTS = "Inputs/salmon.merged.gene_counts.tsv"
GENOME_SIZES = "Inputs/chrNameLength.txt"
WHITELIST_TXT = "Inputs/TRIBE_Huh7_whitelist.txt"
WHITELIST_BED = "Inputs/TRIBE_Huh7_whitelist.bed"

IFIT3_BEDS = Path(f"{wd}Inputs/IFIT3_BEDS")
YTH_BEDS = Path(f"{wd}Inputs/YTH_BEDS")

IFIT3_BED = f"{wd}Inputs/IFIT3_BEDS/IFIT3_TRIBE_HITS.sorted.bed"
IFIT3_CTL_BED = f"{wd}Inputs/IFIT3_BEDS/IFIT3_TRIBE_NS.sorted.bed"

PERMUT_DIR = Path("shuffled")
PERMUT_DIR.mkdir(exist_ok=True)

# Base functions


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1-3. **Make pseudo peaks**
1. **Cluster by a specified interval**
2. **Merge the clusters using bedtools groupby**
3. **Add flanking interval corresponding to number specified in step 1**

All this is wrapped into the called function

In [5]:
help(overlap_scripts.TRIBE_pseudo_peak)

Help on function TRIBE_pseudo_peak in module overlap_scripts:

TRIBE_pseudo_peak(bed, out_parent_dir, dist, genome_file, steps=None)
    Runs clustering and groupby/slop summary to produce pseudo-peak BEDs.
    
    Args:
        bed:           Input BED file or path to BED directory/glob.
        out_parent_dir:Directory under which all output folders will be created.
        dist:          Maximum clustering distance (int).
        genome_file:   Chrom sizes file for bedtools slop.
        steps:         Optional step size for clusters (int or None). If given, runs clusters at step increments.



In [21]:
overlap_scripts.TRIBE_pseudo_peak(IFIT3_BEDS, f'{IFIT3_BEDS}/pseudo_peaks', 250, GENOME_SIZES, steps=50)
overlap_scripts.TRIBE_pseudo_peak(YTH_BEDS, f'{YTH_BEDS}/pseudo_peaks', 250, GENOME_SIZES, steps=50)

Pseudo-peak workflow complete. Results in: /cwork/mt354/MGT494_overlaps_v2/Inputs/IFIT3_BEDS/pseudo_peaks/slopped
Pseudo-peak workflow complete. Results in: /cwork/mt354/MGT494_overlaps_v2/Inputs/YTH_BEDS/pseudo_peaks/slopped


## 4. **Compute overlaps for each size of pseudopeak**

In [6]:
help(overlap_scripts.overlap_diff)

Help on function overlap_diff in module overlap_scripts:

overlap_diff(TRIBE_BED, CONTROL_BED, ref_dir, save_csv=None)
    Calculate overlap counts between TRIBE/CONTROL BEDs and a set of reference BEDs.
    
    For each BED file in ref_dir, computes the number of overlaps with TRIBE_BED and CONTROL_BED,
    and calculates the difference in counts. Results are returned as a pandas DataFrame,
    and can optionally be saved as a CSV file.
    
    Parameters
    ----------
    TRIBE_BED : str or Path
        Path to the TRIBE BED file.
    CONTROL_BED : str or Path
        Path to the CONTROL BED file.
    ref_dir : Path or str
        Directory containing reference BED files to compare against.
    save_csv : bool or str or Path, optional
        If provided, saves the resulting DataFrame to a CSV file.
        If True, saves to a default name in the ref_dir.
        If a string or Path, saves to the given file path.
    
    Returns
    -------
    obs_df : pandas.DataFrame
        D

In [50]:
windows=[50,100,150,200,250]
IFIT3_YTH_obs = Path("IFIT3_YTH/observed")
os.makedirs(IFIT3_YTH_obs,exist_ok=True)
bed_path=f'{IFIT3_BEDS}/pseudo_peaks/slopped'

for x in windows:
    overlap_scripts.overlap_diff(
        f"{bed_path}/{x}/IFIT3_TRIBE_HITS.sorted.{x}.slop.bed",
        f"{bed_path}/{x}/IFIT3_TRIBE_NS.sorted.{x}.slop.bed",
        Path(f"{YTH_BEDS}/pseudo_peaks/slopped/{x}"),
        save_csv=f"{IFIT3_YTH_obs}/{x}/IFIT3_YTH_obs_{x}.csv"
    )

Saved overlap difference results to: IFIT3_YTH/observed/50/IFIT3_YTH_obs_50.csv
Saved overlap difference results to: IFIT3_YTH/observed/100/IFIT3_YTH_obs_100.csv
Saved overlap difference results to: IFIT3_YTH/observed/150/IFIT3_YTH_obs_150.csv
Saved overlap difference results to: IFIT3_YTH/observed/200/IFIT3_YTH_obs_200.csv
Saved overlap difference results to: IFIT3_YTH/observed/250/IFIT3_YTH_obs_250.csv


In [51]:
windows=[50,100,150,200,250]
YTH_IFIT3_obs = Path("YTH_IFIT3/observed")
os.makedirs(YTH_IFIT3_obs,exist_ok=True)
bed_path=f'{YTH_BEDS}/pseudo_peaks/slopped'

for x in windows:
    overlap_scripts.overlap_diff(
        f"{bed_path}/{x}/YTH_TRIBE_HITS.sorted.{x}.slop.bed",
        f"{bed_path}/{x}/YTH_TRIBE_NS.sorted.{x}.slop.bed",
        Path(f"{IFIT3_BEDS}/pseudo_peaks/slopped/{x}"),
        save_csv=f"{YTH_IFIT3_obs}/{x}/YTH_IFIT3_obs_{x}.csv"
    )

Saved overlap difference results to: YTH_IFIT3/observed/50/YTH_IFIT3_obs_50.csv
Saved overlap difference results to: YTH_IFIT3/observed/100/YTH_IFIT3_obs_100.csv
Saved overlap difference results to: YTH_IFIT3/observed/150/YTH_IFIT3_obs_150.csv
Saved overlap difference results to: YTH_IFIT3/observed/200/YTH_IFIT3_obs_200.csv
Saved overlap difference results to: YTH_IFIT3/observed/250/YTH_IFIT3_obs_250.csv


### 5. **Permutation test: Shuffle and compute overlap differences**

At each iteration, shuffle TRIBE and Control "peaks" (restricted to whitelist gene intervals), overlap with filtered RBP peaks, compute difference, and store the null distribution.

In [37]:
%%bash -s "$IFIT3_BEDS" "$GENOME_SIZES" "$WHITELIST_BED" "$YTH_BEDS"

TRIBE_BEDS="${1}/pseudo_peaks/slopped"

for dir in ${TRIBE_BEDS}/*; do
    # Remove the trailing slash from dirname if you want
    dirname="${dir%/}"
    window="$(basename "$dirname")"

    # Set variables (example values)
    N_PERMUT_TOTAL=1000
    CHUNK_SIZE=10
    TRIBE_BED="${dirname}/IFIT3_TRIBE_HITS.sorted.${window}.slop.bed" # Adjust as needed
    CONTROL_BED="${dirname}/IFIT3_TRIBE_NS.sorted.${window}.slop.bed" # Adjust as needed
    GENOME_SIZES=$2
    WHITELIST_BED=$3
    FILTERED_RBP_DIR="${4}/pseudo_peaks/slopped/${window}"
    PERMUT_DIR="IFIT3_YTH/shuffled_${window}" # Adjust as needed
    MERGE_OUT_DIR="IFIT3_YTH/merged_permdiffs_${window}" # Adjust as needed

    # create logs directory if not exists
    mkdir -p logs

    # --- 1. Submit permute chunk job array ---
    perm_jobid=$(sbatch --parsable \
        --array=1-100 \
        permute_chunk_bash.sh \
        -n "$N_PERMUT_TOTAL" \
        -c "$CHUNK_SIZE" \
        -i "$TRIBE_BED" \
        -j "$CONTROL_BED" \
        -g "$GENOME_SIZES" \
        -w "$WHITELIST_BED" \
        -r "$FILTERED_RBP_DIR" \
        -p "$PERMUT_DIR")

    echo "Processing $(basename "$TRIBE_BED") and $(basename "$CONTROL_BED") against ${FILTERED_RBP_DIR}"
    echo "Shuffled sequences and chunks saved to ${PERMUT_DIR}"
    echo "Final permuation differences saved to ${MERGE_OUT_DIR}"
    echo
    echo "Permutation array submitted with job ID: $perm_jobid"


    # --- 2. Prepare RBP name list for the merge array job (if using merge_array.sh) ---
    mkdir -p "$MERGE_OUT_DIR"
    ls "$FILTERED_RBP_DIR"/*.bed | xargs -n1 basename | sed 's/\.bed//' > "${MERGE_OUT_DIR}/rbp_names.txt"
    n_rbps=$(wc -l < "${MERGE_OUT_DIR}/rbp_names.txt")

    # --- 3. Submit merge array (for N RBPs) dependent on permute finishing ---
    merge_jobid=$(sbatch --parsable \
        --dependency=afterok:$perm_jobid \
        --array=1-"$n_rbps" \
        merge_array.sh \
        -p "$PERMUT_DIR" \
        -m "$MERGE_OUT_DIR" \
        -l "${MERGE_OUT_DIR}/rbp_names.txt")

    echo "Merge job array submitted with dependency on permutation job $perm_jobid (merge job id $merge_jobid)."
    echo
    echo

done

Processing IFIT3_TRIBE_HITS.sorted.100.slop.bed and IFIT3_TRIBE_NS.sorted.100.slop.bed against /cwork/mt354/MGT494_overlaps_v2/Inputs/YTH_BEDS/pseudo_peaks/slopped/100
Shuffled sequences and chunks saved to IFIT3_YTH/shuffled_100
Final permuation differences saved to IFIT3_YTH/merged_permdiffs_100

Permutation array submitted with job ID: 32531533
Merge job array submitted with dependency on permutation job 32531533 (merge job id 32531534).


Processing IFIT3_TRIBE_HITS.sorted.150.slop.bed and IFIT3_TRIBE_NS.sorted.150.slop.bed against /cwork/mt354/MGT494_overlaps_v2/Inputs/YTH_BEDS/pseudo_peaks/slopped/150
Shuffled sequences and chunks saved to IFIT3_YTH/shuffled_150
Final permuation differences saved to IFIT3_YTH/merged_permdiffs_150

Permutation array submitted with job ID: 32531535
Merge job array submitted with dependency on permutation job 32531535 (merge job id 32531536).


Processing IFIT3_TRIBE_HITS.sorted.200.slop.bed and IFIT3_TRIBE_NS.sorted.200.slop.bed against /cwork/mt35

In [38]:
%%bash -s "$YTH_BEDS" "$GENOME_SIZES" "$WHITELIST_BED" "$IFIT3_BEDS"

TRIBE_BEDS="${1}/pseudo_peaks/slopped"

for dir in ${TRIBE_BEDS}/*; do
    # Remove the trailing slash from dirname if you want
    dirname="${dir%/}"
    window="$(basename "$dirname")"

    # Set variables (example values)
    N_PERMUT_TOTAL=1000
    CHUNK_SIZE=10
    TRIBE_BED="${dirname}/YTH_TRIBE_HITS.sorted.${window}.slop.bed" # Adjust as needed
    CONTROL_BED="${dirname}/YTH_TRIBE_NS.sorted.${window}.slop.bed" # Adjust as needed
    GENOME_SIZES=$2
    WHITELIST_BED=$3
    FILTERED_RBP_DIR="${4}/pseudo_peaks/slopped/${window}"
    PERMUT_DIR="YTH_IFIT3/shuffled_${window}" # Adjust as needed
    MERGE_OUT_DIR="YTH_IFIT3/merged_permdiffs_${window}" # Adjust as needed

    # create logs directory if not exists
    mkdir -p logs

    # --- 1. Submit permute chunk job array ---
    perm_jobid=$(sbatch --parsable \
        --array=1-100 \
        permute_chunk_bash.sh \
        -n "$N_PERMUT_TOTAL" \
        -c "$CHUNK_SIZE" \
        -i "$TRIBE_BED" \
        -j "$CONTROL_BED" \
        -g "$GENOME_SIZES" \
        -w "$WHITELIST_BED" \
        -r "$FILTERED_RBP_DIR" \
        -p "$PERMUT_DIR")

    echo "Processing $(basename "$TRIBE_BED") and $(basename "$CONTROL_BED") against ${FILTERED_RBP_DIR}"
    echo "Shuffled sequences and chunks saved to ${PERMUT_DIR}"
    echo "Final permuation differences saved to ${MERGE_OUT_DIR}"
    echo
    echo "Permutation array submitted with job ID: $perm_jobid"


    # --- 2. Prepare RBP name list for the merge array job (if using merge_array.sh) ---
    mkdir -p "$MERGE_OUT_DIR"
    ls "$FILTERED_RBP_DIR"/*.bed | xargs -n1 basename | sed 's/\.bed//' > "${MERGE_OUT_DIR}/rbp_names.txt"
    n_rbps=$(wc -l < "${MERGE_OUT_DIR}/rbp_names.txt")

    # --- 3. Submit merge array (for N RBPs) dependent on permute finishing ---
    merge_jobid=$(sbatch --parsable \
        --dependency=afterok:$perm_jobid \
        --array=1-"$n_rbps" \
        merge_array.sh \
        -p "$PERMUT_DIR" \
        -m "$MERGE_OUT_DIR" \
        -l "${MERGE_OUT_DIR}/rbp_names.txt")

    echo "Merge job array submitted with dependency on permutation job $perm_jobid (merge job id $merge_jobid)."
    echo
    echo

done

Processing YTH_TRIBE_HITS.sorted.100.slop.bed and YTH_TRIBE_NS.sorted.100.slop.bed against /cwork/mt354/MGT494_overlaps_v2/Inputs/IFIT3_BEDS/pseudo_peaks/slopped/100
Shuffled sequences and chunks saved to YTH_IFIT3/shuffled_100
Final permuation differences saved to YTH_IFIT3/merged_permdiffs_100

Permutation array submitted with job ID: 32531674
Merge job array submitted with dependency on permutation job 32531674 (merge job id 32531675).


Processing YTH_TRIBE_HITS.sorted.150.slop.bed and YTH_TRIBE_NS.sorted.150.slop.bed against /cwork/mt354/MGT494_overlaps_v2/Inputs/IFIT3_BEDS/pseudo_peaks/slopped/150
Shuffled sequences and chunks saved to YTH_IFIT3/shuffled_150
Final permuation differences saved to YTH_IFIT3/merged_permdiffs_150

Permutation array submitted with job ID: 32531676
Merge job array submitted with dependency on permutation job 32531676 (merge job id 32531699).


Processing YTH_TRIBE_HITS.sorted.200.slop.bed and YTH_TRIBE_NS.sorted.200.slop.bed against /cwork/mt354/MGT494

### 6. **Empirical p-values and multiple testing correction**

In [43]:
os.chdir(wd)

In [53]:
windows=[50,100,150,200,250]
obs_path = Path("IFIT3_YTH/observed")
bed_path=f'{IFIT3_BEDS}/pseudo_peaks/slopped'

for x in windows:
    
    obs_df = pd.read_csv(f'{obs_path}/{x}/IFIT3_YTH_obs_{x}.csv')
    
    overlap_scripts.process_permdiff(
    obs_df,
    f'IFIT3_YTH/merged_permdiffs_{x}',
    f'{bed_path}/{x}/IFIT3_TRIBE_HITS.sorted.{x}.slop.bed',
    f'{bed_path}/{x}/IFIT3_TRIBE_NS.sorted.{x}.slop.bed'
    )

IFIT3_CONTROL_permdiff.csv saved to IFIT3_YTH/merged_permdiffs_50
IFIT3_CONTROL_permdiff.csv saved to IFIT3_YTH/merged_permdiffs_100
IFIT3_CONTROL_permdiff.csv saved to IFIT3_YTH/merged_permdiffs_150
IFIT3_CONTROL_permdiff.csv saved to IFIT3_YTH/merged_permdiffs_200
IFIT3_CONTROL_permdiff.csv saved to IFIT3_YTH/merged_permdiffs_250


In [54]:
windows=[50,100,150,200,250]
obs_path = Path("YTH_IFIT3/observed")
bed_path=f'{YTH_BEDS}/pseudo_peaks/slopped'

for x in windows:
    
    obs_df = pd.read_csv(f'{obs_path}/{x}/YTH_IFIT3_obs_{x}.csv')
    
    overlap_scripts.process_permdiff(
    obs_df,
    f'YTH_IFIT3/merged_permdiffs_{x}',
    f'{bed_path}/{x}/YTH_TRIBE_HITS.sorted.{x}.slop.bed',
    f'{bed_path}/{x}/YTH_TRIBE_NS.sorted.{x}.slop.bed'
    )

YTH_CONTROL_permdiff.csv saved to YTH_IFIT3/merged_permdiffs_50
YTH_CONTROL_permdiff.csv saved to YTH_IFIT3/merged_permdiffs_100
YTH_CONTROL_permdiff.csv saved to YTH_IFIT3/merged_permdiffs_150
YTH_CONTROL_permdiff.csv saved to YTH_IFIT3/merged_permdiffs_200
YTH_CONTROL_permdiff.csv saved to YTH_IFIT3/merged_permdiffs_250
